## Import des bibliothèques et téléchargement des données

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
!pip install -q gdown

In [ ]:
!gdown 1JtAIiOVieSEtc9EiiyZlt3j-4sT9_gmw -O archive.zip
!unzip archive.zip

In [ ]:
import os

tot_image_count = 0
main_directory = 'EuroSAT_RGB'

for subdirectory in os.listdir(main_directory):
    subdirectory_path = os.path.join(main_directory, subdirectory)
    if os.path.isdir(subdirectory_path):
        image_count=0
        for filename in os.listdir(subdirectory_path):
            if filename.endswith(('.png', '.jpg', '.jpeg')):  # Assuming image files have these extensions
                image_count += 1
        print(f"Directory: {subdirectory}, Nb image: {image_count}")
        tot_image_count = tot_image_count+image_count

print(f"Total number of images: {tot_image_count}")

In [ ]:
class2idx={"River":0,
           "SeaLake":1,
           "Highway":2,
           "Forest":3,
           "Industrial":4,
           "HerbaceousVegetation":5,
           "Residential":6,
           "AnnualCrop":7,
           "Pasture":8,
           "PermanentCrop":9}
n_class=len(class2idx)

## On crée une liste avec le nom de fichier et l'étiquette

In [ ]:
subdirectories = [d for d in os.listdir(main_directory) if os.path.isdir(os.path.join(main_directory, d))]
image_paths = []
for subdirectory in subdirectories:
    subdirectory_path = os.path.join(main_directory, subdirectory)
    for filename in os.listdir(subdirectory_path):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append((os.path.join(subdirectory_path, filename),
                                subdirectory,
                                class2idx[subdirectory]))

## Visualisation des données (10 images prise aléatoirement)

In [ ]:
random.shuffle(image_paths)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

displayed_count = 0
for image_path, class_name, idx in image_paths:
    if displayed_count < 10:
            img = Image.open(image_path)
            i=displayed_count//5
            j=displayed_count%5
            axes[i,j].imshow(img)
            axes[i,j].set_title(class_name+" : "+str(idx))
            displayed_count=displayed_count+1
    else:
        break

plt.tight_layout()
plt.show()

In [ ]:
image_paths[0]

In [ ]:
rbg_cha=Image.open(image_paths[0][0]).convert('RGB')

In [ ]:
np.array(rbg_cha)

In [ ]:
transform = transforms.ToTensor()
transform(rbg_cha)

## Questions :
1) Quelle est la taille des données
2) Calculer la valeur maximum, minimum et moyenne pour les données transformées avec `ToTensor` et avec `np.array`
3) Afficher la ditribution des entrées pour les données transformées avec `ToTensor` et avec `np.array`

# Base de données


In [ ]:
# Define dataset class
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, img_paths):
        self.im_paths = img_paths

    def __len__(self):
        return len(self.im_paths)

    def __getitem__(self, idx):
        img_path = self.im_paths[idx][0]
        label = torch.tensor(self.im_paths[idx][2])
        image = transform(np.array(Image.open(img_path).convert('RGB')))
        return image, label

In [ ]:
full_dataset = MyDataset(image_paths)


## Question :
Modifier pour définir un jeu de données de validation et un jeu de test

##Modèle statistique

In [ ]:
model = nn.Sequential(
    # Bloc 1
    nn.Conv2d(3, 16, kernel_size=3, padding=1),  # (batch, 16, 64, 64)
    nn.ReLU(),
    nn.MaxPool2d(2),                              # (batch, 16, 32, 32)

    # Bloc 2
    nn.Conv2d(16, 32, kernel_size=3, padding=1), # (batch, 32, 32, 32)
    nn.ReLU(),
    nn.MaxPool2d(2),                             # (batch, 32, 16, 16)

    # Bloc 3
    nn.Conv2d(32, 64, kernel_size=3, padding=1), # (batch, 64, 16, 16)
    nn.ReLU(),
    nn.MaxPool2d(2),                             # (batch, 64, 8, 8)

    # Classifieur
    nn.Flatten(),                                # (batch, 64*8*8) = (batch, 4096)
    nn.Linear(4096, 128),
    nn.ReLU(),
    nn.Linear(128, n_class),
)

In [ ]:
model = nn.Sequential(nn.Flatten(),
                      nn.Linear(64 * 64 * 3, 256),
                      nn.ReLU(),
                      nn.Linear(256, 64),
                      nn.ReLU(),
                      nn.Linear(64, n_class)
                      )

In [ ]:
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Nb parameters =", pytorch_total_params)

## Question:
Dire à quoi correspondent les deux modèles.

In [ ]:
# Divide with
# torch.utils.data.random_split

# Create data loaders
batch_size = 32
data_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# Define models and optimizers
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#resnet18 = models.resnet18(pretrained=True)
#num_ftrs = resnet18.fc.in_features
#resnet18.fc = nn.Linear(num_ftrs, num_classes)
#resnet18 = resnet18.to(device)
# Reduced learning rate for ResNet18

import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

model = model.to(device)
# Reduced learning rate for EfficientNet-b0

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
def train_model(model, dataloader, optimizer, criterion, num_epochs, model_name):
    model.train()
    train_losses = []
    train_accuracies = []
    best_accuracy = 0.0
    best_model_state = None

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_corrects = 0
        epoch_total = 0
        progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{num_epochs} - {model_name}', unit='batch')
        for inputs, labels in progress_bar:

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * inputs.size(0)
            epoch_corrects += torch.sum(preds == labels.data)
            epoch_total += labels.size(0)

            epoch_accuracy = epoch_corrects.double() / epoch_total
            progress_bar.set_postfix(loss=loss.item(), accuracy=epoch_accuracy.item())

        epoch_loss = epoch_loss / epoch_total
        epoch_accuracy = epoch_corrects.double() / epoch_total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy.item())

        # Evaluate on the test set at the end of each epoch
        #test_loss, test_accuracy, _ = evaluate_model(model, test_loader, criterion)
        #print(f'{model_name} - Epoch {epoch+1} - Train Loss: {epoch_loss:.4f} - Train Acc: {epoch_accuracy:.4f} - Test Loss: {test_loss:.4f} - Test Acc: {test_accuracy:.4f}')
        print(f'{model_name} - Epoch {epoch+1} - Train Loss: {epoch_loss:.4f} - Train Acc: {epoch_accuracy:.4f}')

    return train_losses,train_accuracies

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
train_loss,train_ac=train_model(model,data_loader,optimizer, criterion, 10, "MLP")

## Question sur le MLP
1) Modifier la fonction d'entrainement pour ajouter une évaluation avec les données de validation à la fin de chaque époque
2) Faire la courbe d'apprentissage
3) Explorer le changement de quelques hyperparamètres
4) A l'aide des données test, calculer les performances avec la précision, le F1, le recall et la matrice de confusion.


In [ ]:
from torchvision import models

model = models.resnet18(pretrained=True)

# Remplacer uniquement la dernière couche
model.fc = nn.Linear(model.fc.in_features, n_class)
model = model.to(device)